<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/K21_51_KhonaPlus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/parashar21/main/images/khona21-v0.png) <br>


---



[Prithwis Mukerjee](http://www.yantrajaal.com) ||
Website - [Parashar21](https://parashar21.blogspot.com)<br>

# Khona21 — Interpretive Jyotiṣa

**Khona21** is an experimental AI-assisted Jyotiṣa interpretation platform, developed as a companion to **Parashar21**, which performs the astronomical calculations underlying traditional Indian astrology.

Named after the legendary Bengali astrologer Khona, the platform combines precomputed Rashi and Navamsha charts, Vimshottari Dasha schedules and planetary transits (Gochar) with a curated corpus of Jyotiṣa literature, currently comprising works by **James Braha** and **Harihar Majumdar**.

Rather than relying on hard-coded interpretive rules, Khona21 employs **source-grounded, multipass LLM reasoning** to examine planetary configurations, yogas, cancellations, exceptions and conflicting interpretations. An initial assessment establishes the horoscope's principal characteristics, while subsequent consultations investigate specific questions using the original reference texts and relevant astrological evidence.

This notebook, **k21_KhonaPlus**, implements the consultation workflow, integrating deterministic calculations from Parashar21 with the interpretive capabilities of modern Large Language Models.

**Khona21 explores the convergence of traditional Jyotiṣa knowledge and modern artificial intelligence, combining computational precision with source-grounded reasoning to produce systematic, transparent and auditable astrological interpretations.**

In [2]:
from datetime import datetime
import pytz
print("\033[1m"+'জন্ম-মৃত্যু-বিবাহ, তিন না জানেন বরাহ।'+"\033[0m")
print(datetime.now(pytz.timezone('Asia/Calcutta')))

জন্ম-মৃত্যু-বিবাহ, তিন না জানেন বরাহ।
2026-10-29 20:04:32.322474+05:30


#Setup, Installs, Imports

In [3]:
!pip -q install pypandoc
!pip -q install python-docx
from IPython.display import display, Markdown


!wget -O k21core.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21core.py
!wget -O k21fileops.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21fileops.py
from k21core import *
from k21fileops import *

In [4]:
# ------------------------------------------------
# required for multiple imports of the same module
# ------------------------------------------------
#import k21core
#import importlib

#importlib.reload(k21core)
from k21core import *

In [5]:
# ------------------------------------------------
# Set Parameters
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
model = "gpt-6-luna"

#model = "gpt-6.1-sol"

#model = "gpt-6-astra"

Native_id = "TopGun"

###Access Control
This notebook accesses Google Drive to access files and uses the OpenAI API. Hence it can be used ONLY from the userid of the creator. Others will have to create their own access paths

In [6]:
# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF


## Load Files

In [7]:
!wget -O braha.txt -q "https://raw.githubusercontent.com/prithwis/parashar21/main/Corpus Texts/Text_Extracts/BR_Curated_22.txt"
!wget -O harihar.txt -q "https://raw.githubusercontent.com/prithwis/parashar21/main/Corpus Texts/Text_Extracts/HM_Clean_07.txt"
!wget -O role.txt -q https://raw.githubusercontent.com/prithwis/parashar21/main/RoleFiles/khona261029A.txt

BRAHA   = "/content/braha.txt"
HARIHAR = "/content/harihar.txt"
ROLE    = "/content/role.txt"

In [8]:
#
# Following files from Parashar21 need to be uploaded here
#

CHART = "/content/Chart.txt"
NAVAMSA = "/content/Navamsa_Chart.txt"
DASHA = "/content/Dasha.txt"
GOCHAR = "/content/Gochar_Chart.txt"
GOCHAR_EVENTS = "/content/KaalTransit_261029.txt"

# LLM Access




In [9]:
# ------------------------------------------------
# Define the ROLE
# ------------------------------------------------
#RoleFile = f"{K21_HOME}/Roles/khona261027.txt"
khona21 = readText(ROLE)

/content/role.txt | 6,402 characters


In [10]:
# ------------------------------------------------
# Set the Context
# ------------------------------------------------
#BaseCONTEXT = buildContext(BRAHA, HARIHAR, CHART, NAVAMSA, DASHA)
BaseCONTEXT = getContext(
    BRAHA=BRAHA,
    HARIHAR=HARIHAR,
    CHART=CHART,
    NAVAMSA=NAVAMSA,
    DASHA=DASHA
)

In [11]:
len(BaseCONTEXT)

944654

## the FirstPass

In [12]:
# ------------------------------------------------
# FIRST PASS
#
# ------------------------------------------------

FirstPass_Instruction = """
FIRST PASS — FOUNDATIONAL ASSESSMENT

Prepare a concise, high-level assessment of this horoscope.

Present an overall picture of the native's principal
strengths, weaknesses and significant planetary configurations.

Identify the most important favourable and unfavourable yogas,
sign exchanges, cancellations and mitigating factors,
to the extent that they are evident from the supplied data
and reference texts.

Begin with a simple-language summary, followed by a brief
astrological explanation with relevant source references.

This is an initial orientation, not an exhaustive audit.
Focus on the dominant features of the horoscope rather than
attempting to catalogue every planetary condition or yoga.

Distinguish chart facts from interpretations, and acknowledge
important contradictions or uncertainties.

Do not invent configurations or use rules unsupported by
the supplied reference texts.

Do not make predictions about specific future events.
"""

In [13]:
# ------------------------------------------------
# Assemble the first pass query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(BaseCONTEXT, FirstPass_Instruction)

FirstPass = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)


OPENAI | gpt-6-luna | Effort: medium | In: 222,577 | Cached: 0 | Out: 6,983 | Reasoning: 5,432 | Time: 91.0s


In [ ]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
#MD_FILE = saveAnswer(model, FirstPass)
#display(Markdown(FirstPass))

## Add Gochar and Transit Info

In [14]:
Gochar = readText(GOCHAR)
Transits = readText(GOCHAR_EVENTS)

/content/Gochar_Chart.txt | 4,864 characters
/content/KaalTransit_261029.txt | 5,413 characters


In [15]:
CONTEXT = augContext(
    BaseCONTEXT,
    FIRSTPASS=FirstPass,
    GOCHAR=Gochar,
    TRANSITS=Transits
)

## Konsultation

In [16]:
# ------------------------------------------------
# Ask your question / define the task
# This and subsequent cells can be run multiple times
# ------------------------------------------------
CurrentQuestion = """
What does the native's horoscope indicate about the two years, from November 2026 to October 2028?

Identify the principal favourable and unfavourable developments that may arise during this period, considering professional life, intellectual and creative pursuits, finances, health, family and personal relationships.

Which areas show the greatest promise, and which present the most significant risks or difficulties?

Where possible, identify periods when particular opportunities or difficulties may become more pronounced. Distinguish between strong indications and more speculative possibilities, and explain the astrological reasoning behind each conclusion.

Do not assume that the period will be predominantly favourable or unfavourable. Assess both possibilities independently.
"""

In [17]:
# ------------------------------------------------
# Assemble the query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(CONTEXT, CurrentQuestion)

answer = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6-luna | Effort: medium | In: 227,644 | Cached: 1,259 | Out: 10,814 | Reasoning: 6,670 | Time: 120.7s


In [18]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(model, answer)
# ------------------------------------------------
# Convert markdown to docx for human readability
# Second Parameter is identifier of native, can be replaced by id
# ------------------------------------------------
DOCX_FILE = markdownToDocx(MD_FILE, Native_id, CurrentQuestion, provider,model)

Saved: /content/K21_261029_201145_gpt-6-luna.md
Saved: /content/K21_261029_201145_gpt-6-luna.docx


## Check Credit Status
https://platform.openai.com/settings/organization/billing/credit-grants

https://platform.openai.com/usage


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)